# 第 24 章 · 实时会话：字幕、背压和打断

实时会话至少有控制流、媒体流、字幕流。字幕是可替换快照，音频是有时序的片段，工具请求则需要可靠关联；统一用无限队列会把延迟和取消问题藏起来。

本章运行真实 asyncio 有界队列和字幕 reducer。字节只是本地占位帧；没有麦克风、扬声器、网络、编解码器或真实语音服务。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · partial 覆盖，final 产生待提交候选

字幕输入是完整快照，不是增量拼接。final 后只创建 candidate；真正接回时还要让原 Scheduler 受理，才能称任务已提交。改变 revision 顺序，观察过期帧被丢弃。

In [2]:
m = load_example('24', 'events')
reducer = m.TranscriptReducer('session-1', 1, 'thread-1')
def transcript(revision, text, final=False, attempt=1, speaker='user'):
    return m.Transcript('session-1', attempt, speaker, 'utterance-1', revision, text, final)
assert reducer.receive(transcript(1, '修复')) == 'caption_updated'
assert reducer.receive(transcript(2, '修复 status')) == 'caption_updated'
assert reducer.receive(transcript(1, '旧内容')) == 'stale_revision'
assert reducer.captions[('user', 'utterance-1')].text == '修复 status'
assert reducer.receive(transcript(3, '修复 status 筛选', True)) == 'candidate_pending'
assert len(reducer.candidates) == 1
candidate = next(iter(reducer.candidates.values()))
assert candidate.thread_id == 'thread-1'
print(candidate)

SubmissionCandidate(key=('session-1', 1, 'user', 'utterance-1'), thread_id='thread-1', text='修复 status 筛选')


## 2 · final 不变，旧 attempt 不能混进新会话

同一句 final 重发可以去重，冲突 final 必须暴露。关闭会话后保留 candidate 供核对，但不再接纳事件，也不自动把旧候选重放成新任务。

In [3]:
assert reducer.receive(transcript(3, '修复 status 筛选', True)) == 'duplicate_final'
assert reducer.receive(transcript(4, '晚到 partial')) == 'late_partial'
try:
    reducer.receive(transcript(4, '另一个 final', True))
except ValueError as error:
    assert str(error) == 'conflicting final'
else:
    raise AssertionError('冲突 final 未暴露')
new_reducer = m.TranscriptReducer('session-1', 2, 'thread-1')
assert new_reducer.receive(transcript(5, '旧连接晚到', True, attempt=1)) == 'stale_session'
assert not new_reducer.candidates
reducer.close()
assert reducer.receive(transcript(6, '关闭之后')) == 'closed'
assert len(reducer.candidates) == 1
print({'retained_candidates': len(reducer.candidates), 'new_attempt_candidates': len(new_reducer.candidates)})

{'retained_candidates': 1, 'new_attempt_candidates': 0}


## 3 · 有界队列把背压传回生产者

容量为 1 时第二次 put 必须等待消费者取走第一帧。把 capacity 改为 2，会让“正在阻塞”的断言失败。实验用事件循环让出执行权观察等待状态，避免用长时间 sleep 猜测。

In [4]:
import asyncio
async def backpressure_experiment():
    channel = m.EpochChannel(capacity=1)
    await channel.put(0, b'frame-1')
    producer = asyncio.create_task(channel.put(0, b'frame-2'))
    await asyncio.sleep(0)
    assert not producer.done() and channel.size == 1
    assert await channel.get() == (0, b'frame-1')
    await asyncio.wait_for(producer, timeout=1)
    assert await channel.get() == (0, b'frame-2')
    await channel.close()
    return {'capacity': 1, 'producer_blocked_before_get': True}
await backpressure_experiment()

{'capacity': 1, 'producer_blocked_before_get': True}

## 4 · interrupt 使排队帧与等待写入同时失效

只清空队列不够：正在等待的旧 producer 醒来后可能重新塞入旧音频。epoch 是数据代次，interrupt 在同一条件锁下递增它。恢复产生新 epoch；close 还必须唤醒等待者并返回关闭。

In [5]:
async def interrupt_experiment():
    channel = m.EpochChannel(capacity=1)
    await channel.put(0, b'old-a')
    waiting = asyncio.create_task(channel.put(0, b'old-b'))
    await asyncio.sleep(0)
    epoch, discarded = await channel.interrupt()
    assert (epoch, discarded) == (1, 1)
    try:
        await asyncio.wait_for(waiting, timeout=1)
    except m.StaleEpoch:
        pass
    else:
        raise AssertionError('旧写入应被 epoch fence 拒绝')
    await channel.put(epoch, b'new-a')
    assert await channel.get() == (1, b'new-a')
    reader = asyncio.create_task(channel.get())
    await asyncio.sleep(0)
    await channel.close()
    try:
        await asyncio.wait_for(reader, timeout=1)
    except m.ChannelClosed:
        pass
    else:
        raise AssertionError('close 必须唤醒等待者')
    return {'epoch': epoch, 'discarded': discarded, 'waiting_reader': 'closed'}
await interrupt_experiment()

{'epoch': 1, 'discarded': 1, 'waiting_reader': 'closed'}

## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/core/src/realtime_conversation.rs#L691-L761](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/realtime_conversation.rs#L691-L761)
- [codex-rs/tui/src/chatwidget/realtime.rs#L590-L600](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/tui/src/chatwidget/realtime.rs#L590-L600)
- [codex-rs/realtime-webrtc/src/session.rs#L96-L175](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/realtime-webrtc/src/session.rs#L96-L175)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。